---
## BƯỚC 1: Cài đặt và import thư viện


In [2]:
# ==========================================
# TIKTOK CREATORS DATA AUDIT & SAMPLING FRAME
# Phase GĐ1: Audit & Sampling Design
# ==========================================

import os
import pymongo
import pandas as pd
import numpy as np
from datetime import datetime


---
## BƯỚC 2: Kết nối MongoDB


In [3]:
# Cấu hình kết nối
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")  # Địa chỉ MongoDB
DB_NAME = "tiktok_ads_db"  # Tên database
COLLECTION_NAME = "creators_vn1"  # Tên collection (bảng)

# Thực hiện kết nối
try:
    client = pymongo.MongoClient(MONGO_URI)
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]
    
    # Kiểm tra kết nối
    doc_count = collection.count_documents({})
    print(f"✓ Kết nối thành công!")
    print(f"✓ Database: {DB_NAME}")
    print(f"✓ Collection: {COLLECTION_NAME}")
    print(f"✓ Tổng số bản ghi: {doc_count:,}")
    
except Exception as e:
    print(f"✗ Lỗi kết nối: {e}")
    raise

✓ Kết nối thành công!
✓ Database: tiktok_ads_db
✓ Collection: creators_vn1
✓ Tổng số bản ghi: 9,966


---
## BƯỚC 3: Tải dữ liệu từ MongoDB


In [4]:
# Chọn các trường (cột) cần lấy
fields_to_get = {
    '_id': 1,              # ID MongoDB
    'ID': 1,               # ID creator
    'Name': 1,             # Tên creator
    'Country': 1,          # Quốc gia
    'Followers': 1,        # Số followers
    'Engagement': 1,       # Tỷ lệ tương tác
    'Median Views': 1,     # Lượt xem trung vị
    'Start Price': 1,      # Giá khởi điểm
    'Broadcast Score': 1,  # Điểm phát sóng
    'Collab Score': 1,     # Điểm hợp tác
    'Tags': 1              # Danh mục
}

print("Đang tải dữ liệu từ MongoDB...")

# Lấy dữ liệu
data = list(collection.find({}, fields_to_get))

# Chuyển thành DataFrame
df = pd.DataFrame(data)

print(f"✓ Đã tải {len(df):,} bản ghi")
print(f"✓ Số cột: {len(df.columns)}")

# Đổi tên cột sang tiếng Anh dễ hiểu
df.rename(columns={
    'ID': 'creator_id',
    'Name': 'name',
    'Country': 'country',
    'Followers': 'followers',
    'Engagement': 'engagement',
    'Median Views': 'median_views',
    'Start Price': 'price',
    'Broadcast Score': 'broadcast_score',
    'Collab Score': 'collab_score',
    'Tags': 'category'
}, inplace=True)

# Xem 5 dòng đầu tiên
print("\n5 dòng dữ liệu đầu tiên:")
print(df.head())

Đang tải dữ liệu từ MongoDB...
✓ Đã tải 9,966 bản ghi
✓ Số cột: 11

5 dòng dữ liệu đầu tiên:
              _id broadcast_score collab_score  country engagement followers  \
0     nhut26march            97.9         89.1  Unknown     17.31%      1.5M   
1  maitrithuc2020            95.8         80.6  Unknown      9.41%        2M   
2         mikeden            95.8         81.5  Unknown      6.91%      4.1M   
3         dntminh            92.3         76.9  Unknown     13.47%      2.6M   
4     hoangvinhhh            94.4         78.6  Unknown      6.83%     12.4M   

       creator_id median_views               name                price  \
0     nhut26march       192.8K         Huỳnh Nhựt       52,620,000 VND   
1  maitrithuc2020       845.6K       Mai Trí Thức  Thỏa thuận/Chưa đặt   
2         mikeden       229.5K           Mike Đen        2,700,000 VND   
3         dntminh       670.1K        Vitamin Mèo  Thỏa thuận/Chưa đặt   
4     hoangvinhhh       598.7K  Nguyễn Hoàng Vinh       

In [4]:
df_head = df.head(100)  # Lấy 100 dòng đầu tiên để phân tích
df_head.to_excel("tiktok_creators_head_100.xlsx", index=False)  # Lưu ra file Excel

---
## Làm sạch và chuyển đổi dữ liệu

In [7]:
print("Đang làm sạch dữ liệu...\n")

# ===== Hàm chuyển đổi số followers =====
def convert_followers(value):
    """
    Chuyển '1,5M' -> 1500000
    Chuyển '500K' -> 500000
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip().replace(',', '.')
    
    if 'M' in value:
        return float(value.replace('M', '')) * 1_000_000
    elif 'K' in value:
        return float(value.replace('K', '')) * 1_000
    else:
        try:
            return float(value)
        except:
            return None

# ===== Hàm chuyển đổi tỷ lệ engagement =====
def convert_engagement(value):
    """
    Chuyển '14,06%' -> 14.06
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip().replace('%', '').replace(',', '.')
    try:
        return float(value)
    except:
        return None

# ===== Hàm chuyển đổi giá =====
def convert_price(value):
    """
    Chuyển '12.801.218 VND' -> 12801218
    'Thỏa thuận' -> None
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip()
    
    if 'Thỏa thuận/Chưa đặt' in value or 'Chưa đặt' in value:
        return None
    
    # Loại bỏ 'VND' và các ký tự không phải số
    value = value.replace('VND', '').replace('.', '').replace(',', '').strip()
    try:
        return float(value)
    except:
        return None

# ===== Áp dụng chuyển đổi =====
df['followers_num'] = df['followers'].apply(convert_followers)
df['engagement_num'] = df['engagement'].apply(convert_engagement)
df['median_views_num'] = df['median_views'].apply(convert_followers)
df['price_num'] = df['price'].apply(convert_price)

print("✓ Đã chuyển đổi followers")
print("✓ Đã chuyển đổi engagement")
print("✓ Đã chuyển đổi median_views")
print("✓ Đã chuyển đổi price")

# Xem kết quả
print("\nDữ liệu sau khi chuyển đổi:")
print(df[['name', 'followers', 'followers_num', 'price', 'price_num']].head())

Đang làm sạch dữ liệu...

✓ Đã chuyển đổi followers
✓ Đã chuyển đổi engagement
✓ Đã chuyển đổi median_views
✓ Đã chuyển đổi price

Dữ liệu sau khi chuyển đổi:
                name followers  followers_num                price   price_num
0         Huỳnh Nhựt      1.5M      1500000.0       52,620,000 VND  52620000.0
1       Mai Trí Thức        2M      2000000.0  Thỏa thuận/Chưa đặt         NaN
2           Mike Đen      4.1M      4100000.0        2,700,000 VND   2700000.0
3        Vitamin Mèo      2.6M      2600000.0  Thỏa thuận/Chưa đặt         NaN
4  Nguyễn Hoàng Vinh     12.4M     12400000.0       13,155,000 VND  13155000.0


In [8]:
df.isnull().sum() / len(df)

_id                 0.000000
broadcast_score     0.000000
collab_score        0.000000
country             0.000000
engagement          0.000000
followers           0.000000
creator_id          0.000000
median_views        0.000000
name                0.000000
price               0.000000
category            0.000000
followers_num       0.000000
engagement_num      0.002308
median_views_num    0.002308
price_num           0.588100
dtype: float64

In [10]:
df.describe()

,followers_num,engagement_num,median_views_num,price_num
count,9.966000e+03,9943.000000,9.943000e+03,4.105000e+03
mean,3.292177e+05,5.643836,8.570893e+04,1.395919e+07
std,8.584511e+05,4.918939,3.031269e+05,1.207887e+08
min,3.200000e+03,0.060000,1.100000e+01,1.000000e+00
25%,4.580000e+04,2.540000,1.000000e+04,9.999990e+05
50%,1.158000e+05,4.230000,2.680000e+04,2.000000e+06
75%,2.794500e+05,7.420000,7.215000e+04,5.000000e+06
max,2.120000e+07,94.160000,2.290000e+07,2.631000e+09


In [9]:
df.duplicated(subset=["creator_id"]).sum()

0

In [11]:
df["followers_log"] = np.log1p(df["followers_num"])
df["price_num_log"] = np.log1p(df["price_num"])
df["median_views_num_log"] = np.log1p(df["median_views_num"])


##  Phân loại creators theo mức giá

In [5]:

# Tính ngưỡng phân vị (33% và 67%)
price_33 = df['price_num'].quantile(0.33)
price_67 = df['price_num'].quantile(0.67)

print(f"Ngưỡng 33%: {price_33:,.0f} VND")
print(f"Ngưỡng 67%: {price_67:,.0f} VND")
print()

# Hàm phân loại
def classify_price(price):
    if price < price_33:
        return 'low'
    elif price < price_67:
        return 'mid'
    else:
        return 'high'

# Áp dụng phân loại
df['price_tier'] = df['price_num'].apply(classify_price)

# Thống kê
price_stats = df['price_tier'].value_counts().sort_index()
print("Phân bổ theo mức giá:")
for tier, count in price_stats.items():
    percent = count / len(df) * 100
    print(f"  {tier.upper():8s}: {count:4,} creators ({percent:5.1f}%)")

Ngưỡng 33%: 1,000,000 VND
Ngưỡng 67%: 4,000,000 VND

Phân bổ theo mức giá:
  HIGH    : 7,247 creators ( 72.7%)
  LOW     : 1,027 creators ( 10.3%)
  MID     : 1,692 creators ( 17.0%)


# Phân loại creators theo quy mô

In [6]:
print("Đang phân loại theo quy mô...\n")

# Định nghĩa ngưỡng
MICRO_THRESHOLD = 500_000   # 500K
MID_THRESHOLD = 2_000_000   # 2M
print(f"Ngưỡng MICRO: < {MICRO_THRESHOLD:,} followers")
print(f"Ngưỡng MID: {MICRO_THRESHOLD:,} - {MID_THRESHOLD:,} followers")
print(f"Ngưỡng MACRO: > {MID_THRESHOLD:,} followers")
# Hàm phân loại
def classify_size(followers):
    if followers < MICRO_THRESHOLD:
        return 'micro'
    elif followers < MID_THRESHOLD:
        return 'mid'
    else:
        return 'macro'

# Áp dụng phân loại
df['size_tier'] = df['followers_num'].apply(classify_size)

# Thống kê
size_stats = df['size_tier'].value_counts().sort_index()
print("Phân bổ theo quy mô:")
for tier, count in size_stats.items():
    percent = count / len(df) * 100
    print(f"  {tier.upper():8s}: {count:4,} creators ({percent:5.1f}%)")

Đang phân loại theo quy mô...

Ngưỡng MICRO: < 500,000 followers
Ngưỡng MID: 500,000 - 2,000,000 followers
Ngưỡng MACRO: > 2,000,000 followers
Phân bổ theo quy mô:
  MACRO   :  273 creators (  2.7%)
  MICRO   : 8,554 creators ( 85.8%)
  MID     : 1,139 creators ( 11.4%)


# Phân tích theo danh mục (Category)

In [7]:
# ===== DEMO: Hiểu cấu trúc dữ liệu category =====
print("Ví dụ dữ liệu category của 3 creators đầu tiên:\n")

for i in range(min(3, len(df))):
    name = df.iloc[i]['name']
    cats = df.iloc[i]['category']
    
    print(f"{i+1}. {name}")
    print(f"   Type: {type(cats)}")
    print(f"   Value: {cats}")
    
    # Nếu là string có dấu phẩy
    if isinstance(cats, str) and ',' in cats:
        cat_list = [c.strip() for c in cats.split(',')]
        print(f"   Số danh mục: {len(cat_list)}")
        for cat in cat_list[:3]:  # Chỉ hiển thị 3 danh mục đầu
            print(f"      - {cat}")
    print()


Ví dụ dữ liệu category của 3 creators đầu tiên:

1. Huỳnh Nhựt
   Type: <class 'str'>
   Value: Oral Care, Movies & TV, Comedy, News & Entertainment, E-Commerce (Non-app), Beauty & Personal Care
   Số danh mục: 6
      - Oral Care
      - Movies & TV
      - Comedy

2. Mai Trí Thức
   Type: <class 'str'>
   Value: Comedy

3. Mike Đen
   Type: <class 'str'>
   Value: Non-video Games, Games, Video Games, Apps, Household Products
   Số danh mục: 5
      - Non-video Games
      - Games
      - Video Games



In [8]:
print("Phân tích theo danh mục nội dung...\n")

# ===== XỬ LÝ DANH MỤC DẠNG LIST =====

# Bước 1: Tách từng danh mục từ list
all_categories = []

for categories in df['category']:
    # Kiểm tra nếu là string (có dấu phẩy) thì split
    if isinstance(categories, str):
        cat_list = [c.strip() for c in categories.split(',')]
        all_categories.extend(cat_list)
    # Nếu là list thì thêm trực tiếp
    elif isinstance(categories, list):
        all_categories.extend(categories)

# Bước 2: Đếm tần suất từng danh mục
from collections import Counter
category_counter = Counter(all_categories)

# Bước 3: Lấy top 20 danh mục phổ biến
top_20 = category_counter.most_common(20)

print("Top 20 danh mục phổ biến nhất:")
print("(Lưu ý: 1 creator có thể có nhiều danh mục)\n")

total_tags = sum(category_counter.values())
for i, (cat, count) in enumerate(top_20, 1):
    percent = count / total_tags * 100
    print(f"{i:2d}. {count:4,} lượt ({percent:4.1f}%) - {cat}")

print(f"\nTổng số danh mục khác nhau: {len(category_counter):,}")
print(f"Tổng số lượt gắn tag: {total_tags:,}")
print(f"Trung bình mỗi creator có: {total_tags/len(df):.1f} danh mục")

Phân tích theo danh mục nội dung...

Top 20 danh mục phổ biến nhất:
(Lưu ý: 1 creator có thể có nhiều danh mục)

 1. 1,727 lượt ( 7.6%) - Daily Life
 2. 1,513 lượt ( 6.7%) - Lip Syncing
 3. 1,347 lượt ( 5.9%) - Beauty Tutorials & Tips
 4. 1,183 lượt ( 5.2%) - Dance
 5.  961 lượt ( 4.2%) - Selfie
 6.  900 lượt ( 4.0%) - Video Games
 7.  856 lượt ( 3.8%) - Outfits
 8.  823 lượt ( 3.6%) - Music
 9.  809 lượt ( 3.6%) - Beauty & Personal Care
10.  647 lượt ( 2.8%) - News & Entertainment
11.  624 lượt ( 2.7%) - Movies & TV
12.  566 lượt ( 2.5%) - Comedy
13.  500 lượt ( 2.2%) - Family
14.  470 lượt ( 2.1%) - Restaurant Exploration
15.  442 lượt ( 1.9%) - Love & Romantic Relationships
16.  441 lượt ( 1.9%) - Animation & Cosplay
17.  389 lượt ( 1.7%) - Tech Products & Tests
18.  377 lượt ( 1.7%) - Mukbang & Food Tasting
19.  351 lượt ( 1.5%) - Apparel & Accessories
20.  326 lượt ( 1.4%) - Cooking & Recipes

Tổng số danh mục khác nhau: 108
Tổng số lượt gắn tag: 22,719
Trung bình mỗi creator có: 

Nhóm nội dung chiếm ưu thế:

- (1+4+5+14)    Beauty & Fashion : ~3,768 lượt (~17%)
- (3+6+10+12)   Entertainment    : ~3,252 lượt (~15%)
- (2+16+19+20)  Lifestyle        : ~2,409 lượt (~11%)

# Trực quan sâu về những kênh có quy mô cao